In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import hashlib
import secrets
import numpy as np
import matplotlib.pyplot as plt

from ecc import *

# Módulo 5: ECDSA — Firmas Digitales

**Dónde estamos:** En el Módulo 4 vimos el cifrado ElGamal — Bob enmascara un
mensaje con un secreto compartido derivado de la clave pública de Alice, y Alice
lo desenmascara con su clave privada. ECDSA es la misma familia de ideas, pero en
lugar de *cifrar* un mensaje, estamos *demostrando que lo escribimos* — sin
revelar la clave privada.

ElGamal también tenía un esquema de firmas (no solo cifrado). DSA lo refinó, y
ECDSA lo trasplantó a curvas elípticas — el mismo movimiento que Koblitz y Miller
hicieron para el intercambio de claves.

## La función unidireccional se dispara dos veces

Cada transacción de Bitcoin usa la función unidireccional $P = d \times G$ en dos
momentos diferentes:

**Momento 1 — generación de clave** (ocurrió una vez, cuando se creó la billetera):
elegiste un secreto $d$, calculaste $P = d \times G$, publicaste $P$. Nadie puede
revertir $P$ a $d$. Ese es el alias.

**Momento 2 — firma** (ocurriendo ahora mismo, para esta transacción): eliges un
$k$ aleatorio fresco, calculas $R = k \times G$. La función unidireccional se
dispara *otra vez*. $R$ es público, pero $k$ está escondido dentro. Luego entrelazas
$k$, tu clave privada $d$, y el hash del mensaje en un solo número $s$. La firma
es $(r, s)$ donde $r$ es la coordenada x de $R$.

## 5.1 Firma — qué hace realmente el firmante

### La situación concreta

Alice tiene 0.5 BTC en una dirección. Esa dirección se deriva de su clave
pública $P$, que se deriva de su clave privada $d$. Quiere enviar 0.3 BTC a Bob.

Crea una transacción: "Mover 0.3 BTC de dirección(P) a dirección(Bob)."
Esta transacción se transmitirá a toda la red — todos la verán. La red necesita
verificar: **¿la persona que creó esta transacción realmente controla los fondos
en dirección(P)?**

Alice no puede mostrar su clave privada $d$ — toda la red la vería y robaría
sus 0.2 BTC restantes. En su lugar, produce una **firma**: un par de números
$(r, s)$ que demuestran que conoce $d$ sin revelarlo.

```
  Alice conoce: clave privada d, clave pública P = d×G, mensaje a firmar

  Paso 1: Hash del mensaje
           z = SHA-256(mensaje)

  Paso 2: Elegir un nonce aleatorio k
           Fresco, secreto, nunca reutilizado. (Veremos por qué en 5.4.)

  Paso 3: Calcular R = k × G
           La función unidireccional se dispara. R es un punto de la curva.
           r = coordenada x de R mod N

  Paso 4: Calcular s = k⁻¹ · (z + r·d) mod N
           Esta es la ecuación central. Ata:
           - z (el mensaje — prueba QUÉ se firmó)
           - r (el compromiso del nonce — prueba ESTA sesión de firma)
           - d (la clave privada — prueba QUIÉN firmó)

  Paso 5: Resultado: firma (r, s)
           Dos números. Eso es todo. Clave privada no revelada.
```

## 5.2 Verificación — qué hace realmente el verificador

Alice transmite su transacción a la red junto con su firma $(r, s)$ y su clave
pública $P$.

Cada nodo que recibe esta transacción necesita responder una pregunta:
**¿el dueño de esta clave pública realmente autorizó esta transacción?**

```
  El verificador conoce: clave pública P, mensaje, firma (r, s)

  Paso 1: Hash del mensaje
           z = SHA-256(mensaje)

  Paso 2: Calcular s⁻¹ mod N

  Paso 3: Calcular dos "pesos"
           u₁ = z · s⁻¹ mod N    (cuánto de G)
           u₂ = r · s⁻¹ mod N    (cuánto de P)

  Paso 4: Calcular R' = u₁ × G + u₂ × P
           Si la firma es válida, R' debería ser igual al R del firmante.

  Paso 5: Verificar: ¿R'_x mod N es igual a r?
           Si sí → firma válida. Alice tiene la clave privada.
           Si no → firma falsificada o corrupta.
```

**La magia:** El verificador nunca conoció $k$ o $d$, pero combinando $G$
(que todos conocen) y $P$ (que contiene $d$ escondido dentro), ponderados por
piezas de la firma, reconstruye el mismo punto $R$ que el firmante calculó.

In [ ]:
import hashlib

def ecdsa_sign(message: bytes, private_key: int) -> tuple:
    """Generate ECDSA signature (r, s)."""
    z = int.from_bytes(hashlib.sha256(message).digest(), 'big')
    
    while True:
        k = secrets.randbelow(SECP_N - 1) + 1
        R = scalar_mult(k, G)
        r = R.x % SECP_N
        if r == 0:
            continue
        
        k_inv = pow(k, SECP_N - 2, SECP_N)
        s = (k_inv * (z + r * private_key)) % SECP_N
        if s == 0:
            continue
        
        return (r, s)

def ecdsa_verify(message: bytes, signature: tuple, public_key: Point) -> bool:
    """Verify ECDSA signature."""
    r, s = signature
    z = int.from_bytes(hashlib.sha256(message).digest(), 'big')
    
    s_inv = pow(s, SECP_N - 2, SECP_N)
    u1 = (z * s_inv) % SECP_N
    u2 = (r * s_inv) % SECP_N
    
    R_prime = point_add(scalar_mult(u1, G), scalar_mult(u2, public_key))
    
    return R_prime.x % SECP_N == r

# Generate key pair
d = secrets.randbelow(SECP_N - 1) + 1
P = scalar_mult(d, G)

# Sign a message
msg = b"Hello, Bitcoin!"
sig = ecdsa_sign(msg, d)
r, s = sig

print("=== Demostración ECDSA ===")
print(f"Mensaje: {msg.decode()}")
print(f"Private key d: {hex(d)[:20]}...")
print(f"Public key P:  {P}")
print(f"\nFirma:")
print(f"  r = {hex(r)[:20]}...")
print(f"  s = {hex(s)[:20]}...")

# Verify
valid = ecdsa_verify(msg, sig, P)
print(f"\nVerificación: {valid}  ✓")

# Tamper with message
tampered = b"Hello, Bitcorn!"
valid_tampered = ecdsa_verify(tampered, sig, P)
print(f"Verificación de msg alterado: {valid_tampered}  ✗ (correctly rejected)")

# Wrong key
wrong_key = scalar_mult(secrets.randbelow(SECP_N - 1) + 1, G)
valid_wrong = ecdsa_verify(msg, sig, wrong_key)
print(f"Verificación con clave incorrecta: {valid_wrong}  ✗ (correctly rejected)")

## 5.3 Por Qué Funciona la Verificación — La Demostración

Dijimos que la clave privada "se cancela" durante la verificación. Veámoslo
algebraicamente — cada paso usa las propiedades de grupo del Módulo 1.

El verificador calcula $R' = u_1 \times G + u_2 \times P$. Si la firma es
válida, $R'$ debería ser igual al $R = kG$ del firmante. Demostrémoslo:

$$R' = u_1 G + u_2 P$$

Sustituir $u_1 = z/s$, $u_2 = r/s$, y $P = dG$:

$$R' = \frac{z}{s} G + \frac{r}{s}(dG)$$

Factorizar $G$ y $1/s$ (esto usa **conmutatividad** y **asociatividad**):

$$= \frac{z + rd}{s} G$$

De la ecuación de firma: $s = k^{-1}(z + rd)$, entonces $(z + rd)/s = k$:

$$R' = k G = R \quad \checkmark$$

La $d$ estaba escondida dentro de $P = dG$. La ecuación de verificación la
extrajo justo lo suficiente para cancelarse con la $d$ incrustada en $s$ —
pero nunca expuso $d$ en sí.

## 5.4 La Catástrofe del Nonce

Mira la ecuación de firma otra vez: $s = k^{-1}(z + r \cdot d) \bmod N$.

La clave privada $d$ está protegida por el nonce $k$. Si $k$ es aleatorio y
secreto, no hay forma de aislar $d$ de una sola ecuación — dos incógnitas,
una ecuación.

**Pero ¿qué pasa si reutilizas $k$?**

Si el mismo nonce $k$ se usa para dos mensajes diferentes, el atacante obtiene
dos ecuaciones con el mismo $k$:

$$s_1 = k^{-1}(z_1 + r \cdot d) \qquad s_2 = k^{-1}(z_2 + r \cdot d)$$

Restar — $d$ se cancela:

$$s_1 - s_2 = k^{-1}(z_1 - z_2)$$

Ahora resuelve para $k$:

$$k = \frac{z_1 - z_2}{s_1 - s_2} \bmod N$$

Una vez que $k$ es conocido, sustituye en cualquier ecuación y resuelve para $d$:

$$d = r^{-1}(s \cdot k - z) \bmod N$$

**Fin del juego.** La clave privada queda expuesta.

### Esto realmente ocurrió

**Sony PS3 (2010):** Sony firmó cada actualización de firmware de PS3 con ECDSA.
Usaron el mismo $k$ para cada firma — literalmente una constante. El grupo hacker
fail0verflow notó que dos firmas tenían el mismo valor $r$ (mismo $k$ significa
mismo $R = kG$ significa mismo $r$), aplicaron el álgebra de arriba, y extrajeron
la clave maestra de firma de Sony.

**Billeteras Bitcoin de Android (2013):** Un bug en la implementación de
`SecureRandom` de Android causó que algunas apps de billetera Bitcoin reutilizaran
nonces. Los atacantes monitorearon la blockchain, encontraron pares de transacciones
de la misma dirección con el mismo valor $r$, extrajeron las claves privadas y
robaron los bitcoin.

### La solución: nonces determinísticos (RFC 6979)

No dependas de generadores de números aleatorios. En su lugar, calcula $k$
determinísticamente:

$$k = \text{HMAC-SHA256}(\text{clave privada}, \text{hash del mensaje})$$

Misma clave + mismo mensaje → mismo $k$ → misma firma (reproducible).
Mensaje diferente → $k$ diferente → reutilización imposible.

In [ ]:
# Demonstration: nonce reuse catastrophe

victim_priv = secrets.randbelow(SECP_N - 1) + 1
victim_pub = scalar_mult(victim_priv, G)

# Victim signs two messages with the SAME nonce (fatal mistake)
k_reused = secrets.randbelow(SECP_N - 1) + 1
R_k = scalar_mult(k_reused, G)
r_val = R_k.x % SECP_N
k_inv = pow(k_reused, SECP_N - 2, SECP_N)

msg1 = b"Transfer 1 BTC to Alice"
msg2 = b"Transfer 2 BTC to Bob"
z1 = int.from_bytes(hashlib.sha256(msg1).digest(), 'big')
z2 = int.from_bytes(hashlib.sha256(msg2).digest(), 'big')

s1 = (k_inv * (z1 + r_val * victim_priv)) % SECP_N
s2 = (k_inv * (z2 + r_val * victim_priv)) % SECP_N

print("=== Ataque por Reutilización de Nonce ===")
print(f"El atacante ve dos firmas con el mismo r:")
print(f"  sig1: r = {hex(r_val)[:16]}..., s1 = {hex(s1)[:16]}...")
print(f"  sig2: r = {hex(r_val)[:16]}..., s2 = {hex(s2)[:16]}...")
print(f"  Mismo r → se usó el mismo nonce k!")

# Attacker recovers k
k_recovered = ((z1 - z2) * pow(s1 - s2, SECP_N - 2, SECP_N)) % SECP_N
print(f"\nEl atacante recupera k: {k_recovered == k_reused}")

# Attacker recovers private key
d_recovered = (pow(r_val, SECP_N - 2, SECP_N) * (s1 * k_recovered - z1)) % SECP_N
print(f"El atacante recupera la clave privada: {d_recovered == victim_priv}")
print(f"\n⚠️  NUNCA reutilices un nonce. Bitcoin usa RFC 6979 (k determinístico).")

## 5.5 De Matemáticas a Bytes — Codificación DER

Las matemáticas de ECDSA producen $(r, s)$ como enteros grandes. Pero las
transacciones de Bitcoin son flujos de bytes. ¿Cómo se convierten las
matemáticas en formato de cable?

### Codificación DER

Las firmas ECDSA usan **DER** (Distinguished Encoding Rules), un formato
ASN.1 de longitud variable:

```
0x30 || largo_total || 0x02 || largo_r || bytes_r || 0x02 || largo_s || bytes_s
```

Cada entero ($r$ y $s$) se codifica como una cadena de bytes de longitud
variable — si el bit alto está activo, se antepone un byte de relleno `0x00`
para mantenerlo positivo. Esto hace que las firmas ECDSA sean de **longitud
variable** (típicamente 71-73 bytes), más un byte de flag sighash al final.

### Normalización Low-S (BIP 62)

Bitcoin requiere $s \leq N/2$. Si $s > N/2$, reemplazarlo con $N - s$.

¿Por qué? Porque tanto $(r, s)$ como $(r, N-s)$ pasan la verificación — ambas
son firmas válidas. Eso significa que cualquiera podría tomar tu transacción,
cambiar $s$ por $N-s$ y retransmitirla. La transacción seguiría siendo válida
pero tendría un **hash diferente**. Esto es **maleabilidad de transacciones**.

La regla low-S elimina la ambigüedad: solo uno de los dos valores de $s$ es
aceptado, así nadie puede cambiarlo.

En el Módulo 6 veremos cómo Schnorr elimina ambos problemas (64 bytes fijos,
sin maleabilidad por diseño).

In [ ]:
def der_encode_integer(value: int) -> bytes:
    """Encode a positive integer in DER format (0x02 || length || bytes)."""
    b = value.to_bytes((value.bit_length() + 7) // 8, 'big')
    if b[0] & 0x80:
        b = b'\x00' + b
    return bytes([0x02, len(b)]) + b

def der_encode_signature(r: int, s: int) -> bytes:
    """DER-encode an ECDSA signature: 0x30 || total_len || r_der || s_der."""
    r_der = der_encode_integer(r)
    s_der = der_encode_integer(s)
    body = r_der + s_der
    return bytes([0x30, len(body)]) + body

def low_s_normalize(s: int, N: int) -> int:
    """BIP 62: if s > N/2, replace with N - s."""
    if s > N // 2:
        return N - s
    return s

# Use the signature we generated in section 5.2
# r and s are the two components of any ECDSA signature
r_example = 0xDEADBEEF_CAFEBABE_12345678_9ABCDEF0_DEADBEEF_CAFEBABE_12345678_9ABCDEF0
s_example = 0xFEDCBA98_76543210_FEDCBA98_76543210_FEDCBA98_76543210_FEDCBA98_76543210

print("=== Codificación DER (ECDSA) ===\n")
der_sig = der_encode_signature(r_example, s_example)
print(f"r = {hex(r_example)[:20]}...")
print(f"s = {hex(s_example)[:20]}...")
print(f"\nDER-encoded: {der_sig.hex()}")
print(f"DER length:  {len(der_sig)} bytes (variable)")
print(f"\nByte breakdown:")
print(f"  0x30       = SEQUENCE tag")
print(f"  0x{der_sig[1]:02x}       = total body length ({der_sig[1]} bytes)")
print(f"  0x02       = INTEGER tag (r)")
print(f"  0x{der_sig[3]:02x}       = r length ({der_sig[3]} bytes)")
print(f"  {der_sig[4:4+der_sig[3]].hex()[:40]}... = r value")
r_end = 4 + der_sig[3]
print(f"  0x02       = INTEGER tag (s)")
print(f"  0x{der_sig[r_end+1]:02x}       = s length ({der_sig[r_end+1]} bytes)")
print(f"  {der_sig[r_end+2:].hex()[:40]}... = s value")

# With sighash byte (SIGHASH_ALL = 0x01)
der_with_sighash = der_sig + bytes([0x01])
print(f"\n+ sighash byte (0x01) = {len(der_with_sighash)} bytes total on the wire")

# Low-S normalization
print(f"\n=== Normalización Low-S (BIP 62) ===\n")
s_high = SECP_N - 1
s_low = low_s_normalize(s_high, SECP_N)
print(f"Original s:   {hex(s_high)[:20]}... (> N/2)")
print(f"Normalized s: {hex(s_low)[:20]}... (= N - s)")
print(f"s > N/2: {s_high > SECP_N // 2} → replaced to prevent malleability")

print(f"\nECDSA DER signature: {len(der_with_sighash)} bytes on the wire (variable)")
print(f"In Module 6 we'll see Schnorr: fixed 64 bytes, no DER needed.")